# ISOM 835 · Session 10 — Segments, Structure & Anomalies
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 23 · Prof. Hasan Arslan**

Learning without labels: RFM segmentation with k-means, choosing k honestly, hierarchical clustering and DBSCAN, PCA and UMAP for seeing high-dimensional data, and Isolation Forest for the transactions that don't belong.

> **Frame the task.** *Unit:* one customer · *No target* — evaluation is usefulness and stability · *Decision:* one marketing action per segment, and which transactions to send to fraud review.

In [ ]:
import pandas as pd, numpy as np, io, zipfile, urllib.request
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## 1. Load Online Retail II (UCI) — with a fallback
1,067,371 transactions from a UK gift retailer, Dec 2009 – Dec 2011. The UCI zip contains one Excel file with two sheets. If the download fails, the fallback builds a realistic synthetic RFM table so every cell still runs.

In [ ]:
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip', timeout=120).read()))
    xlsx = [n for n in z.namelist() if n.endswith('.xlsx')][0]
    sheets = pd.read_excel(io.BytesIO(z.read(xlsx)), sheet_name=None)
    tx = pd.concat(sheets.values(), ignore_index=True)
    REAL = True
except Exception as e:
    print('download failed →', type(e).__name__, '— using synthetic transactions')
    rng = np.random.default_rng(835); n_cust = 4000
    seg = rng.choice(4, n_cust, p=[.2, .35, .2, .25]); rows = []
    for cid in range(n_cust):
        n_inv = int(rng.poisson([12, 4, 6, 1][seg[cid]]) + 1)
        for i in range(n_inv):
            rows.append({'Invoice': f'{cid}-{i}', 'Customer ID': cid, 'Quantity': int(rng.integers(1, 30)), 'Price': float(rng.lognormal(1.2, 0.7)),
                         'InvoiceDate': pd.Timestamp('2011-12-09') - pd.Timedelta(days=int(rng.exponential([20, 60, 200, 300][seg[cid]])) + int(rng.integers(0, 10)))})
    tx = pd.DataFrame(rows); REAL = False
print(tx.shape, tx.columns.tolist())

In [ ]:
tx = tx.dropna(subset=['Customer ID'])
tx = tx[~tx['Invoice'].astype(str).str.startswith('C') & (tx['Quantity'] > 0) & (tx['Price'] > 0)]
tx['Amount'] = tx['Quantity'] * tx['Price']
snapshot = tx['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = tx.groupby('Customer ID').agg(Recency=('InvoiceDate', lambda d: (snapshot - d.max()).days), Frequency=('Invoice', 'nunique'), Monetary=('Amount', 'sum'))
print(rfm.shape); rfm.describe().round(1)

## 2. Scale like you mean it
k-means minimizes Euclidean distance: unscaled, a $10,000 spend difference is ten thousand times a one-day recency difference. Log the skewed columns, then standardize.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 5))
for j, c in enumerate(rfm.columns):
    rfm[c].plot(kind='hist', bins=40, ax=ax[0, j], color='#7c8cff', title=f'{c} (raw)')
    (np.log1p(rfm[c]) if c != 'Recency' else rfm[c]).plot(kind='hist', bins=40, ax=ax[1, j], color='#2ee6c5', title=f'{c} ({"log" if c != "Recency" else "raw"})')
plt.tight_layout(); plt.show()
Z = StandardScaler().fit_transform(np.column_stack([rfm['Recency'], np.log1p(rfm['Frequency']), np.log1p(rfm['Monetary'])]))

## 3. How many segments? Elbow, silhouette, judgment

In [ ]:
ks = range(2, 10); inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=835).fit(Z)
    inertia.append(km.inertia_); sil.append(silhouette_score(Z, km.labels_, sample_size=3000, random_state=835))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4)); ax[0].plot(list(ks), inertia, 'o-', color='#2ee6c5'); ax[0].set_title('elbow: inertia vs k'); ax[1].plot(list(ks), sil, 'o-', color='#f5a524'); ax[1].set_title('silhouette vs k'); plt.show()
print(pd.DataFrame({'k': list(ks), 'inertia': np.round(inertia, 0), 'silhouette': np.round(sil, 3)}).to_string(index=False))

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=835).fit(Z)
rfm['segment'] = km.labels_
profile = rfm.groupby('segment').agg(n=('Recency', 'size'), Recency=('Recency', 'mean'), Frequency=('Frequency', 'mean'), Monetary=('Monetary', 'mean')).round(1)
def name(r):
    if r.Frequency >= profile.Frequency.quantile(.75) and r.Recency <= profile.Recency.quantile(.5): return 'Champions'
    if r.Recency >= profile.Recency.quantile(.75): return 'Hibernating'
    if r.Frequency >= profile.Frequency.median(): return 'At risk' if r.Recency > profile.Recency.median() else 'Loyal'
    return 'New / occasional'
profile['name'] = profile.apply(name, axis=1); profile

A segmentation is only as good as the four sentences that follow it: *Champions* → early access; *Loyal* → cross-sell; *At risk* → win-back offer; *Hibernating* → nothing (the offer costs more than they're worth).

## 4. Other shapes: hierarchical, DBSCAN, Gaussian mixtures

In [ ]:
S = pd.DataFrame(Z, columns=['R', 'logF', 'logM']).sample(min(3000, len(Z)), random_state=835)
agg = AgglomerativeClustering(n_clusters=4, linkage='ward').fit(S)
db = DBSCAN(eps=0.35, min_samples=15).fit(S)
gmm = GaussianMixture(4, random_state=835).fit(S)
print(f'agglomerative silhouette {silhouette_score(S, agg.labels_):.3f}')
print(f'DBSCAN: {len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)} clusters, {(db.labels_ == -1).mean():.1%} labeled noise (free anomaly detector)')
resp = gmm.predict_proba(S); print(f'GMM: {(resp.max(axis=1) < 0.7).mean():.1%} of customers have no segment with >70% membership — soft assignments')

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
plt.figure(figsize=(9, 3.2)); dendrogram(linkage(S.sample(400, random_state=1), 'ward'), no_labels=True, color_threshold=8); plt.title('dendrogram (400 customers): cut at any height → any k'); plt.show()

## 5. Seeing high dimensions: PCA and UMAP
PCA keeps variance (and its loadings name the axes); UMAP keeps neighborhoods (prettier, less trustworthy distances). Cluster on scaled features, plot on the embedding.

In [ ]:
pca = PCA(2).fit(Z); P = pca.transform(Z)
print('explained variance:', pca.explained_variance_ratio_.round(3)); print(pd.DataFrame(pca.components_, columns=['Recency', 'logFreq', 'logMon'], index=['PC1', 'PC2']).round(2))
plt.figure(figsize=(6, 4.5)); plt.scatter(P[:, 0], P[:, 1], c=rfm['segment'], cmap='tab10', s=5, alpha=0.6); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('segments on the PCA plane'); plt.show()

In [ ]:
# OPTIONAL — pip install umap-learn
try:
    import umap
    U = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=835).fit_transform(Z[:5000])
    plt.figure(figsize=(6, 4.5)); plt.scatter(U[:, 0], U[:, 1], c=rfm['segment'].values[:5000], cmap='tab10', s=5, alpha=0.6); plt.title('UMAP: neighborhoods preserved, distances not'); plt.show()
except ImportError:
    print('umap-learn not installed')

## 6. Anomalies: Isolation Forest
Random trees isolate outliers in a few splits and normal points in many; the average path length is the anomaly score. A simulated card-transaction stream with 1% fraud and no labels.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
Xf, yf = make_classification(n_samples=20000, n_features=12, n_informative=6, weights=[0.99, 0.01], class_sep=1.8, flip_y=0.002, random_state=835)
iso = IsolationForest(n_estimators=300, contamination=0.02, random_state=835).fit(Xf)        # contamination = the alert rate the fraud team can review
score = -iso.score_samples(Xf)                                                               # higher = more anomalous
print(f'no labels used. ROC-AUC vs. hidden fraud label {roc_auc_score(yf, score):.3f}   PR-AUC {average_precision_score(yf, score):.3f}')
top = np.argsort(-score)[:int(0.05 * len(score))]; print(f'top 5% of scores contain {yf[top].sum() / yf.sum():.0%} of the fraud')

## 7. Your turn
1. **k = 5.** Re-run the profile table with five segments. Which segment split, and would marketing act on the new one differently?
2. **Country cut.** (Real data only) Compute segment shares for UK vs. non-UK customers. Does the segmentation travel?
3. **Alert budget.** The fraud team can review 200 cases a day out of 20,000. Set `contamination` accordingly and report how much fraud the top 200 captures.

In [ ]:
# Your turn — work here

## What we learned tonight
- **k-means finds compact clusters of similar size — whether or not they exist.** Scale (and log) first; use k-means++ and `n_init`; let elbow + silhouette inform k and the business decide.
- **PCA keeps variance, UMAP keeps neighborhoods.** Cluster on features, plot on the embedding.
- **Isolation Forest scores by how easy a point is to isolate** — anomaly detection without labels; the alert rate is a business choice.

**HW5** (due Dec 7): Track A (segmentation, tonight) or Track B (forecasting, next week).